## 0. 加载 Qwen 模型

本 Notebook 使用 **Qwen2.5-7B-Instruct**（通过 ModelScope 加载）作为真实 LLM 后端，替代原有的模拟 LLM。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

### 安装依赖（首次运行需要）

```bash
pip install modelscope torch transformers
```


In [ ]:
# ============================================================
# 加载 Qwen 模型（通过 ModelScope）
# ============================================================
# 如果没有 GPU 或显存不足，可将模型 ID 改为 Qwen/Qwen2.5-3B-Instruct
# ============================================================

import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer


class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装类"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测 GPU / CPU
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("[QwenLLM] 模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """调用模型进行对话"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """重置对话历史"""
        self.messages = []


# 初始化 QwenLLM 实例
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
print("\n模型就绪，可以开始使用了。")

# 01 - RAG Agent：检索增强生成智能体

## 学习目标

- 理解 RAG（检索增强生成）的核心原理
- 掌握文档加载、分块、嵌入和检索的流程
- 学习构建 RAG Agent 的完整流程
- 实现一个支持知识库问答的 RAG Agent

---

## 1. RAG 概述

### 1.1 什么是 RAG？

**RAG（Retrieval-Augmented Generation）** 是一种将信息检索与文本生成结合的技术。它让 LLM 能够：

- **访问外部知识**：不仅依赖训练数据，还能查询外部文档
- **减少幻觉**：基于检索到的真实信息生成回答
- **保持更新**：知识库可以随时更新，无需重新训练模型

### 1.2 RAG 工作流程

```
┌─────────────────────────────────────────────────────────────┐
│                     RAG 工作流程                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 文档加载        2. 文本分块        3. 向量化             │
│  ┌─────────┐      ┌─────────┐      ┌─────────┐             │
│  │ 文档A   │─────►│  chunk  │─────►│ 向量    │             │
│  │ 文档B   │      │  chunk  │      │ 向量    │             │
│  │ 文档C   │      │  chunk  │      │ 向量    │             │
│  └─────────┘      └─────────┘      └─────────┘             │
│                                          │                  │
│                                          ▼                  │
│                                    ┌───────────┐            │
│                                    │ 向量数据库 │            │
│                                    └─────┬─────┘            │
│                                          │                  │
│  用户提问 ──► 查询向量化 ──► 相似度检索 ──► 相关文档        │
│                                              │              │
│                                              ▼              │
│                                        ┌─────────┐          │
│                                        │ LLM生成 │          │
│                                        │  回答   │          │
│                                        └─────────┘          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 1.3 RAG vs 传统 LLM

| 特性 | 传统 LLM | RAG |
|------|---------|-----|
| 知识来源 | 训练数据 | 训练数据 + 外部文档 |
| 知识更新 | 需要重新训练 | 实时更新知识库 |
| 幻觉问题 | 较严重 | 显著减少 |
| 可解释性 | 低 | 高（可查看引用来源） |
| 成本 | 高（大模型训练） | 低（只需检索） |

---

## 2. RAG 核心组件

### 2.1 文档加载（Document Loader）

```python
# 常见文档加载方式
from langchain.document_loaders import TextLoader, PDFLoader, WebBaseLoader

# 加载文本文件
loader = TextLoader("document.txt")
documents = loader.load()

# 加载网页
loader = WebBaseLoader("https://example.com/article")
documents = loader.load()
```

### 2.2 文本分块（Text Splitter）

为什么需要分块？
- 文档通常太长，超过 LLM 的上下文限制
- 细粒度分块提高检索精度

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # 每个块的大小
    chunk_overlap=200,    # 块之间的重叠
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(documents)
```

**分块策略对比**：

| 策略 | 优点 | 缺点 |
|------|------|------|
| 固定大小 | 简单 | 可能切断语义 |
| 递归字符 | 保持段落完整 | 块大小不均 |
| 语义分块 | 语义完整 | 计算复杂 |
| 按句子 | 粒度细 | 上下文可能不足 |

### 2.3 嵌入模型（Embedding Model）

将文本转换为向量表示：

```python
from langchain.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

# 将文本转换为向量
text = "AI Agent 是一种能够自主执行任务的智能系统"
vector = embeddings.embed_query(text)

print(f"向量维度: {len(vector)}")  # 通常是 1536 维
```

### 2.4 向量数据库（Vector Store）

存储和检索向量：

```python
from langchain.vectorstores import Chroma, FAISS

# 创建向量数据库
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# 相似度检索
results = vectorstore.similarity_search("什么是 AI Agent?", k=3)
```

**常见向量数据库对比**：

| 数据库 | 特点 | 适用场景 |
|--------|------|----------|
| Chroma | 轻量、易用 | 本地开发 |
| FAISS | Facebook出品、高效 | 大规模检索 |
| Pinecone | 托管服务 | 生产环境 |
| Weaviate | 开源、功能丰富 | 企业应用 |
| Milvus | 分布式 | 大规模部署 |

---

## 3. 动手实现：Mock RAG 系统

下面我们实现一个简化版的 RAG 系统：

In [ ]:
import numpy as np
from typing import List, Dict, Tuple
import re

class MockDocument:
    """模拟文档"""
    
    def __init__(self, content: str, metadata: Dict = None):
        self.page_content = content
        self.metadata = metadata or {}
    
    def __repr__(self):
        return f"MockDocument({self.page_content[:50]}...)"

class MockTextSplitter:
    """模拟文本分块器"""
    
    def __init__(self, chunk_size: int = 100, chunk_overlap: int = 20):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def split_text(self, text: str) -> List[str]:
        """将文本分块"""
        # 按句子分割
        sentences = re.split(r'(?<=[。！？.!?])\s+', text)
        
        chunks = []
        current_chunk = ""
        
        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= self.chunk_size:
                current_chunk += sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(current_chunk)
        
        return chunks
    
    def split_documents(self, documents: List[MockDocument]) -> List[MockDocument]:
        """分割文档"""
        chunks = []
        for doc in documents:
            texts = self.split_text(doc.page_content)
            for i, text in enumerate(texts):
                metadata = {**doc.metadata, "chunk_index": i}
                chunks.append(MockDocument(text, metadata))
        return chunks

# 测试文档和分块
documents = [
    MockDocument("""
    AI Agent（人工智能代理）是一种能够感知环境、做出决策并执行动作的智能系统。
    与传统的软件程序不同，AI Agent 具有自主性，能够在没有人类直接干预的情况下完成任务。
    AI Agent 的核心组件包括：感知模块、推理引擎、行动模块和记忆系统。
    """, {"source": "agent_intro"}),
    
    MockDocument("""
    ReAct（Reasoning + Acting）是一种将推理和行动结合的 Agent 架构。
    它通过交替进行思考（Thought）和行动（Action）来解决复杂问题。
    ReAct 的优势在于能够利用外部工具获取信息，并基于这些信息进行推理。
    """, {"source": "react_arch"}),
    
    MockDocument("""
    向量数据库是 RAG 系统的核心组件，用于存储和检索文本的向量表示。
    常见的向量数据库包括 Chroma、FAISS、Pinecone 等。
    向量数据库通过相似度搜索找到与查询最相关的文档片段。
    """, {"source": "vector_db"})
]

splitter = MockTextSplitter(chunk_size=80, chunk_overlap=10)
chunks = splitter.split_documents(documents)

print(f"原始文档数: {len(documents)}")
print(f"分块后数量: {len(chunks)}")
print("\n分块结果:")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}:")
    print(f"  内容: {chunk.page_content[:60]}...")
    print(f"  元数据: {chunk.metadata}")

In [ ]:
class MockEmbedding:
    """模拟嵌入模型"""
    
    def __init__(self, dim: int = 128):
        self.dim = dim
        np.random.seed(42)
    
    def _text_to_vector(self, text: str) -> np.ndarray:
        """将文本转换为向量（基于词频的简化版）"""
        # 使用简单的哈希方法生成确定性向量
        words = text.lower().split()
        vector = np.zeros(self.dim)
        
        for word in words:
            # 使用单词的哈希值确定位置
            hash_val = hash(word) % self.dim
            vector[hash_val] += 1.0
        
        # 归一化
        norm = np.linalg.norm(vector)
        if norm > 0:
            vector = vector / norm
        
        return vector
    
    def embed_documents(self, texts: List[str]) -> List[np.ndarray]:
        """嵌入多个文档"""
        return [self._text_to_vector(text) for text in texts]
    
    def embed_query(self, text: str) -> np.ndarray:
        """嵌入查询"""
        return self._text_to_vector(text)

class MockVectorStore:
    """模拟向量数据库"""
    
    def __init__(self, embedding_model: MockEmbedding):
        self.embedding_model = embedding_model
        self.documents: List[MockDocument] = []
        self.vectors: List[np.ndarray] = []
    
    def add_documents(self, documents: List[MockDocument]):
        """添加文档"""
        texts = [doc.page_content for doc in documents]
        vectors = self.embedding_model.embed_documents(texts)
        
        self.documents.extend(documents)
        self.vectors.extend(vectors)
        
        print(f"✅ 已添加 {len(documents)} 个文档到向量数据库")
    
    def similarity_search(self, query: str, k: int = 3) -> List[MockDocument]:
        """相似度搜索"""
        if not self.vectors:
            return []
        
        # 嵌入查询
        query_vector = self.embedding_model.embed_query(query)
        
        # 计算相似度（余弦相似度）
        similarities = []
        for vec in self.vectors:
            similarity = np.dot(query_vector, vec)
            similarities.append(similarity)
        
        # 获取最相似的 k 个文档
        top_k_indices = np.argsort(similarities)[-k:][::-1]
        
        results = []
        for idx in top_k_indices:
            doc = self.documents[idx]
            score = similarities[idx]
            # 添加相似度分数到元数据
            doc.metadata["score"] = float(score)
            results.append(doc)
        
        return results

# 创建嵌入模型和向量数据库
embedding = MockEmbedding(dim=128)
vectorstore = MockVectorStore(embedding)

# 添加文档
vectorstore.add_documents(chunks)

# 测试检索
query = "什么是 AI Agent?"
print(f"\n🔍 查询: {query}\n")

results = vectorstore.similarity_search(query, k=2)
print("检索结果:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. [相似度: {doc.metadata['score']:.4f}]")
    print(f"   来源: {doc.metadata['source']}")
    print(f"   内容: {doc.page_content[:80]}...")

In [ ]:
# ---- 无模型时的备选方案：MockLLM（模拟 LLM）----
# class MockLLM:
#     def __init__(self):
#         self.knowledge_base = {
#             "agent": "AI Agent 是一种能够感知环境、做出决策并执行动作的智能系统。",
#             "react": "ReAct 是一种将推理和行动结合的 Agent 架构。",
#             "rag": "RAG（检索增强生成）结合了信息检索和文本生成技术。",
#             "vector": "向量数据库用于存储和检索文本的向量表示。"
#         }
#     def generate(self, prompt: str, context: str = "") -> str:
#         if context:
#             return f"基于检索到的信息，我来回答你的问题：\n\n{context}\n\n根据以上信息，{prompt}\n\n这是一个基于检索结果生成的回答。"
#         else:
#             return f"根据我的知识：\n\n{prompt}\n\n（注意：这是基于训练数据的回答，可能不包含最新信息）"

class RAGAgent:
    """RAG Agent（使用 QwenLLM 进行回答生成）"""
    
    def __init__(self, vectorstore, llm_model, top_k: int = 3):
        self.vectorstore = vectorstore
        self.llm = llm_model  # QwenLLM 实例
        self.top_k = top_k
        self.query_history = []
    
    def query(self, question: str) -> Dict:
        """处理查询"""
        print(f"\n用户问题: {question}")
        print("="*60)
        
        # 1. 检索相关文档
        print("\n步骤1: 检索相关文档...")
        retrieved_docs = self.vectorstore.similarity_search(question, k=self.top_k)
        
        if not retrieved_docs:
            return {
                "question": question,
                "answer": "未找到相关信息",
                "sources": []
            }
        
        print(f"   找到 {len(retrieved_docs)} 个相关文档")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"   {i}. [{doc.metadata['source']}] 相似度: {doc.metadata['score']:.4f}")
        
        # 2. 构建上下文
        print("\n步骤2: 构建上下文...")
        context = "\n\n".join([
            f"[文档 {i+1}] {doc.page_content}"
            for i, doc in enumerate(retrieved_docs)
        ])
        print(f"   上下文长度: {len(context)} 字符")
        
        # 3. 使用 QwenLLM 生成回答
        print("\n步骤3: 使用 QwenLLM 生成回答...")
        system_prompt = (
            "你是一个知识库问答助手。请根据提供的参考文档内容回答用户问题。"
            "回答要准确、简洁，并注明信息来源。"
            "如果文档中没有相关信息，请如实说明。"
        )
        user_msg = f"参考文档：\n{context}\n\n用户问题：{question}"
        answer = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=512, temperature=0.7)
        
        # 记录查询历史
        result = {
            "question": question,
            "answer": answer,
            "sources": [
                {
                    "content": doc.page_content[:100] + "...",
                    "source": doc.metadata["source"],
                    "score": doc.metadata["score"]
                }
                for doc in retrieved_docs
            ],
            "retrieved_count": len(retrieved_docs)
        }
        self.query_history.append(result)
        
        return result
    
    def print_answer(self, result: Dict):
        """打印回答"""
        print("\n" + "="*60)
        print("回答:")
        print("="*60)
        print(result["answer"])
        
        print("\n" + "-"*60)
        print("参考来源:")
        for i, source in enumerate(result["sources"], 1):
            print(f"  {i}. {source['source']} (相似度: {source['score']:.4f})")

# 创建 RAG Agent（使用 QwenLLM 实例）
rag_agent = RAGAgent(vectorstore, llm, top_k=2)

# 测试查询
result = rag_agent.query("什么是 AI Agent?")
rag_agent.print_answer(result)

In [ ]:
# 更多查询示例
questions = [
    "ReAct 架构有什么特点?",
    "向量数据库的作用是什么?",
    "AI Agent 和传统程序有什么区别?"
]

for question in questions:
    result = rag_agent.query(question)
    rag_agent.print_answer(result)
    print("\n" + "█"*60 + "\n")

---

## 4. RAG 优化技巧

### 4.1 查询重写（Query Rewriting）

```python
class QueryRewriter:
    """查询重写器"""
    
    def rewrite(self, query: str) -> List[str]:
        """将查询重写为多个相关查询"""
        variations = [query]
        
        # 添加同义词变体
        if "AI" in query:
            variations.append(query.replace("AI", "人工智能"))
        
        # 添加疑问词变体
        if query.endswith("?"):
            variations.append(query[:-1] + "是什么")
        
        return list(set(variations))
```

### 4.2 重排序（Re-ranking）

```python
class Reranker:
    """重排序器"""
    
    def rerank(self, query: str, documents: List[MockDocument]) -> List[MockDocument]:
        """对检索结果重排序"""
        # 基于关键词匹配度进行重排序
        query_words = set(query.lower().split())
        
        scored_docs = []
        for doc in documents:
            doc_words = set(doc.page_content.lower().split())
            overlap = len(query_words & doc_words)
            scored_docs.append((doc, overlap))
        
        # 按匹配度排序
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored_docs]
```

### 4.3 混合检索（Hybrid Search）

结合向量检索和关键词检索：

```python
class HybridRetriever:
    """混合检索器"""
    
    def __init__(self, vectorstore, keyword_index):
        self.vectorstore = vectorstore
        self.keyword_index = keyword_index
    
    def retrieve(self, query: str, k: int = 5) -> List[MockDocument]:
        """混合检索"""
        # 向量检索
        vector_results = self.vectorstore.similarity_search(query, k=k)
        
        # 关键词检索
        keyword_results = self.keyword_index.search(query, k=k)
        
        # 融合结果（去重并排序）
        all_results = vector_results + keyword_results
        seen = set()
        unique_results = []
        for doc in all_results:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_results.append(doc)
        
        return unique_results[:k]
```

---

## 5. 完整 RAG 系统实现

下面我们实现一个更完整的 RAG 系统，包含所有优化技巧：

In [ ]:
class AdvancedRAGAgent:
    """高级 RAG Agent（使用 QwenLLM）"""
    
    def __init__(self, vectorstore, llm_model):
        self.vectorstore = vectorstore
        self.llm = llm_model  # QwenLLM 实例
        self.query_rewriter = QueryRewriter()
        self.reranker = Reranker()
        self.query_history = []
    
    def query(self, question: str, use_rewrite: bool = True, use_rerank: bool = True) -> Dict:
        """处理查询（带优化）"""
        print(f"\n用户问题: {question}")
        print("="*60)
        
        # 1. 查询重写
        if use_rewrite:
            print("\n步骤1: 查询重写...")
            queries = self.query_rewriter.rewrite(question)
            print(f"   重写为 {len(queries)} 个查询:")
            for q in queries:
                print(f"   - {q}")
        else:
            queries = [question]
        
        # 2. 多查询检索
        print("\n步骤2: 检索相关文档...")
        all_docs = []
        for q in queries:
            docs = self.vectorstore.similarity_search(q, k=3)
            all_docs.extend(docs)
        
        # 去重
        seen = set()
        unique_docs = []
        for doc in all_docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                unique_docs.append(doc)
        
        print(f"   找到 {len(unique_docs)} 个唯一相关文档")
        
        # 3. 重排序
        if use_rerank and unique_docs:
            print("\n步骤3: 重排序...")
            ranked_docs = self.reranker.rerank(question, unique_docs)
            print("   重排序完成")
        else:
            ranked_docs = unique_docs
        
        # 选择 top-k
        top_k = 3
        final_docs = ranked_docs[:top_k]
        
        # 4. 构建上下文
        print("\n步骤4: 构建上下文...")
        context = "\n\n".join([
            f"[文档 {i+1}] 来源: {doc.metadata['source']}\n{doc.page_content}"
            for i, doc in enumerate(final_docs)
        ])
        
        # 5. 使用 QwenLLM 生成回答
        print("\n步骤5: 使用 QwenLLM 生成回答...")
        system_prompt = (
            "你是一个高级知识库问答助手。请根据提供的参考文档内容回答用户问题。"
            "回答要准确、简洁，并注明信息来源。"
        )
        user_msg = f"参考文档：\n{context}\n\n用户问题：{question}"
        answer = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=512, temperature=0.7)
        
        result = {
            "question": question,
            "answer": answer,
            "sources": [
                {
                    "content": doc.page_content[:80] + "...",
                    "source": doc.metadata["source"],
                    "score": doc.metadata.get("score", 0)
                }
                for doc in final_docs
            ],
            "retrieved_count": len(unique_docs),
            "final_count": len(final_docs)
        }
        self.query_history.append(result)
        
        return result
    
    def print_answer(self, result: Dict):
        """打印回答"""
        print("\n" + "="*60)
        print("回答:")
        print("="*60)
        print(result["answer"])
        
        print("\n" + "-"*60)
        print(f"参考来源 (从 {result['retrieved_count']} 个文档中筛选出 {result['final_count']} 个):")
        for i, source in enumerate(result["sources"], 1):
            print(f"  {i}. {source['source']} (相似度: {source['score']:.4f})")

class QueryRewriter:
    """查询重写器"""
    
    def rewrite(self, query: str) -> List[str]:
        """将查询重写为多个相关查询"""
        variations = [query]
        
        # 添加同义词变体
        if "AI" in query:
            variations.append(query.replace("AI", "人工智能"))
        
        if "Agent" in query:
            variations.append(query.replace("Agent", "代理"))
        
        return list(set(variations))

class Reranker:
    """重排序器"""
    
    def rerank(self, query: str, documents) -> list:
        """对检索结果重排序"""
        query_words = set(query.lower().split())
        
        scored_docs = []
        for doc in documents:
            doc_words = set(doc.page_content.lower().split())
            overlap = len(query_words & doc_words)
            scored_docs.append((doc, overlap))
        
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored_docs]

# 创建高级 RAG Agent（使用 QwenLLM 实例）
advanced_rag = AdvancedRAGAgent(vectorstore, llm)

# 测试高级查询
result = advanced_rag.query("AI Agent 是什么?", use_rewrite=True, use_rerank=True)
advanced_rag.print_answer(result)

---

## 6. RAG 评估指标

### 6.1 检索质量评估

| 指标 | 说明 | 计算方式 |
|------|------|----------|
| **Precision@K** | 前K个结果中相关的比例 | 相关文档数 / K |
| **Recall@K** | 相关文档被检索到的比例 | 检索到的相关文档 / 总相关文档 |
| **MRR** | 第一个相关文档的排名倒数 | 1 / 第一个相关文档的排名 |
| **NDCG** | 考虑文档相关度的排序质量 | 加权排序得分 |

### 6.2 生成质量评估

| 指标 | 说明 |
|------|------|
| **Faithfulness** | 回答是否忠实于检索到的文档 |
| **Answer Relevance** | 回答是否与问题相关 |
| **Context Precision** | 使用的上下文是否精确 |
| **Context Recall** | 是否使用了所有相关上下文 |

---

## 7. 小结

### 核心要点

1. **RAG** 结合了检索和生成，让 LLM 能够访问外部知识
2. **核心流程**：文档加载 → 分块 → 嵌入 → 存储 → 检索 → 生成
3. **关键组件**：文档加载器、文本分块器、嵌入模型、向量数据库
4. **优化技巧**：查询重写、重排序、混合检索
5. **评估指标**：检索质量（Precision、Recall）和生成质量（Faithfulness）

### RAG vs Fine-tuning

| 方面 | RAG | Fine-tuning |
|------|-----|-------------|
| 知识更新 | 实时 | 需要重新训练 |
| 成本 | 低 | 高 |
| 灵活性 | 高 | 低 |
| 适用场景 | 频繁更新的知识 | 固定领域知识 |

### 下一步

- [02_tool_use_advanced.ipynb](02_tool_use_advanced.ipynb) - 高级工具使用
- [../04_projects/00_knowledge_base_qa.ipynb](../04_projects/00_knowledge_base_qa.ipynb) - 知识库问答项目

---

## 参考资源

- [LangChain RAG 教程](https://python.langchain.com/docs/use_cases/question_answering/)
- [LlamaIndex RAG](https://docs.llamaindex.ai/en/stable/getting_started/concepts.html)
- [RAG Survey Paper](https://arxiv.org/abs/2312.10997)
- [向量数据库对比](https://python.langchain.com/docs/integrations/vectorstores/)